# Capital Bikeshare 2019: Members vs Casual Riders (SDI Python Module)

About 861,000 Washington DC bike share rides from 2019, joined with hourly weather and US holidays.

**Business question:** How do casual riders use the bikes differently from members, and what does that mean for turning casual riders into members?

Plan:
1. Exploratory Data Analysis I (first look)
2. Cleaning
3. Exploratory Data Analysis II (KPIs)
4. Visualization
5. Recommendations

In [ ]:
# Some questions we may answer
# Locationwise customer footprint
# Which days of the week/time of the day/month have the most rides?
# Which bike stations have the most rides?
# Average duration of bike rides
# casual/member breakdown and their bike use?
# Duration wrt member type
# impact of weather conditions

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# chart colors: one fixed color per rider type, used in every chart
COLORS = {'Member': '#2a78d6', 'Casual': '#eb6834'}

In [ ]:
df = pd.read_csv('/content/Bikeshare Data.csv')
df.tail()

## 1. Exploratory Data Analysis I

In [ ]:
# number of rows & cols
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# breakdown of members vs casual riders (%)
df['Member type'].value_counts(normalize=True) * 100

In [ ]:
# Duration is in seconds
print('Min duration (s):', df['Duration'].min())
print('Max duration (s):', df['Duration'].max())
print('Unique start stations:', df['Start station'].nunique())
print('Unique end stations:', df['End station'].nunique())

In [ ]:
# data quality issues spotted in the first look:
# - 3 leftover index columns (Unnamed: 0, 0.1, 0.2)
# - station number minimum is 0, which is not a real station
# - max duration is 86,305 seconds (about 24 hours), likely bikes that were never docked
# - holiday is blank on normal days
# - dates are stored as text
print('Rows with station number 0:', ((df['Start station number'] == 0) | (df['End station number'] == 0)).sum())
print('Rides over 3 hours:', (df['Duration'] > 3 * 60 * 60).sum())

## 2. Cleaning

In [ ]:
rows_start = len(df)

# drop the 3 leftover index columns
df = df.drop(columns=['Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0'])

# convert seconds to minutes
df['Duration (min)'] = df['Duration'] / 60

# change start/end date to datetime
df['Start date'] = pd.to_datetime(df['Start date'])
df['End date'] = pd.to_datetime(df['End date'])

# fill blank holidays
df['holiday'] = df['holiday'].fillna('Not a Holiday')

# isHoliday is 0/1, make it True/False
df['isHoliday'] = df['isHoliday'].astype(bool)

In [ ]:
# remove rides with station number 0 (not a real station)
df = df[(df['Start station number'] != 0) & (df['End station number'] != 0)]

# station numbers are whole numbers, not decimals
df['Start station number'] = df['Start station number'].astype(int)
df['End station number'] = df['End station number'].astype(int)

In [ ]:
# Decision: remove rides longer than 3 hours.
# A normal trip is minutes long; rides of up to 24 hours are most likely bikes that were not docked,
# and they would pull the casual average up. Print the count so the decision is documented.
long_rides = df['Duration (min)'] > 180
print('Rides removed (over 3 hours):', long_rides.sum())
df = df[~long_rides]

In [ ]:
# check & delete duplicates
print('Duplicate rows:', df.duplicated().sum())
df = df.drop_duplicates()

print('Rows before cleaning:', rows_start)
print('Rows after cleaning:', len(df))
print('Rows removed:', rows_start - len(df))

In [ ]:
# helper columns for time analysis
# note: start times are rounded to the hour in this dataset, so hour of day is the finest level we can use
df['Hour'] = df['Start date'].dt.hour
df['Day of week'] = df['Start date'].dt.day_name()
df['Month'] = df['Start date'].dt.month_name()
df['Weekend'] = df['Start date'].dt.dayofweek >= 5
df.info()

## 3. Exploratory Data Analysis II: KPIs

In [ ]:
# KPI 1: share of rides by rider type
share = df['Member type'].value_counts(normalize=True) * 100
share.round(1)

In [ ]:
# KPI 2: average and median ride length (minutes)
# median is shown too because ride lengths are skewed by a few long rides
df.groupby('Member type')['Duration (min)'].agg(['mean', 'median']).round(1)

In [ ]:
# KPI 3: weekend share of rides for each rider type (%)
(df.groupby('Member type')['Weekend'].mean() * 100).round(1)

In [ ]:
# KPI 4: average temperature (°C) during rides
df.groupby('Member type')['temperature_2m'].mean().round(1)

In [ ]:
# top 10 start stations for casual riders (where to market memberships)
df[df['Member type'] == 'Casual']['Start station'].value_counts().head(10)

In [ ]:
# top 10 start stations for members
df[df['Member type'] == 'Member']['Start station'].value_counts().head(10)

## 4. Visualization

In [ ]:
# Chart 1: average ride length by rider type
avg = df.groupby('Member type')['Duration (min)'].mean().reindex(['Member', 'Casual'])

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(avg.index, avg.values, color=[COLORS[t] for t in avg.index], width=0.5)
ax.bar_label(bars, fmt='%.1f min', padding=3)
ax.set_title('Average ride length by rider type')
ax.set_ylabel('Minutes')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 2: rides by hour of day (commute peaks vs leisure)
by_hour = df.groupby(['Hour', 'Member type']).size().unstack()

fig, ax = plt.subplots(figsize=(9, 4))
for t in ['Member', 'Casual']:
    ax.plot(by_hour.index, by_hour[t], color=COLORS[t], linewidth=2, label=t)
ax.set_title('Rides by hour of day')
ax.set_xlabel('Hour of day (0 = midnight)')
ax.set_ylabel('Number of rides')
ax.set_xticks(range(0, 24, 2))
ax.grid(axis='y', alpha=0.3)
ax.legend(frameon=False, loc='upper left', bbox_to_anchor=(1, 1))
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 3: share of each group's rides by day of week (%), so the two groups are comparable
days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
by_day = df.groupby(['Member type', 'Day of week']).size().unstack(0)
by_day = (by_day / by_day.sum() * 100).reindex(days)

fig, ax = plt.subplots(figsize=(9, 4))
x = range(len(days))
w = 0.38
ax.bar([i - w/2 for i in x], by_day['Member'], width=w, color=COLORS['Member'], label='Member')
ax.bar([i + w/2 for i in x], by_day['Casual'], width=w, color=COLORS['Casual'], label='Casual')
ax.set_xticks(list(x))
ax.set_xticklabels([d[:3] for d in days])
ax.set_title('Share of rides by day of week')
ax.set_ylabel('% of that group\'s rides')
ax.grid(axis='y', alpha=0.3)
ax.legend(frameon=False, loc='upper left', bbox_to_anchor=(1, 1))
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 4: rides by month (seasonality)
months = ['January', 'February', 'March', 'April', 'May', 'June', 'July',
          'August', 'September', 'October', 'November', 'December']
by_month = df.groupby(['Month', 'Member type']).size().unstack().reindex(months)

fig, ax = plt.subplots(figsize=(9, 4))
for t in ['Member', 'Casual']:
    ax.plot([m[:3] for m in months], by_month[t], color=COLORS[t], linewidth=2, marker='o', markersize=4, label=t)
ax.set_title('Rides by month, 2019')
ax.set_ylabel('Number of rides')
ax.grid(axis='y', alpha=0.3)
ax.legend(frameon=False, loc='upper left', bbox_to_anchor=(1, 1))
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## 5. Recommendations / Final thoughts

In [ ]:
# summary of the KPIs behind the recommendations
summary = pd.DataFrame({
    'Share of rides (%)': share.round(1),
    'Avg ride (min)': df.groupby('Member type')['Duration (min)'].mean().round(1),
    'Weekend rides (%)': (df.groupby('Member type')['Weekend'].mean() * 100).round(1),
    'Avg temp (°C)': df.groupby('Member type')['temperature_2m'].mean().round(1),
})
summary

**What the data shows** (check each point against the KPI table above after running the notebook):

- Members take most of the rides (about 79% before cleaning), with short trips that peak at commute hours. They use the bikes for getting to work.
- Casual riders take far longer rides (about 3 times as long before cleaning), lean toward weekends, and ride in warmer weather. They use the bikes for leisure.

**Recommendations:**

1. **Target casual riders at their top 10 start stations** (see the table above) with membership offers, since that's where they already are.
2. **Launch a seasonal or weekend membership** that fits how casual riders actually ride, instead of only pushing the commuter-style annual plan.
3. **Time campaigns for late spring and summer**, when casual riding peaks.

**Limitations:**

- Start times are rounded to the hour, so trip times can't be checked to the minute.
- Rides over 3 hours were removed as likely undocked bikes; this was a judgment call, documented above.
- The data covers 2019 only, so the patterns may have changed since.